In [ ]:
import pymc as pm
import arviz as az
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

n = 100

# compare 2 models where glof is ignored vs not


# i am god
base_flow = np.random.normal(20, 5, n)
glof_indicator = np.random.binomial(1, 0.05, n)  # 5% chance of GLOF warning
actual_flow = base_flow + glof_indicator * np.random.normal(50, 10, n)

# i am a humble man

with pm.Model() as model_naive:

    mu = pm.Normal("mu", mu=20, sigma=10)
    sigma = pm.HalfNormal("sigma", sigma=15)
    y= pm.Normal("y", observed=actual_flow, mu = mu, sigma=sigma)

    prior_checks = pm.sample_prior_predictive(draws=1000, random_seed=42)


    trace_naive = pm.sample(draws=2000, tune=1000, chains=2, random_seed=42, progressbar=False)

    # Add log-likelihood post-sampling for safety and consistency
    pm.compute_log_likelihood(trace_naive)

    # Add posterior predictive data without idata_kwargs

    # cuz asking engine to do 2 things simult
    # 1. simulate fake data
    # 2. evaluate alignment of trace with real data, log likelihood has nothing to do with fake data
    pm.sample_posterior_predictive(trace_naive, extend_inferencedata=True, random_seed=42, progressbar=False)

In [ ]:
az.plot_ppc(prior_checks, group="prior", kind="kde", alpha=0.8)
plt.title("Prior Predictive")
plt.tight_layout()
plt.show()

az.plot_trace(trace_naive, var_names=["mu", "sigma"])
plt.tight_layout()
plt.show()

az.plot_ppc(trace_naive, group="posterior", kind="kde", alpha=0.8)
plt.title("Posterior Predictive")
plt.tight_layout()
plt.show()

print(az.summary(trace_naive, var_names=["mu", "sigma"]))

# our posterior predictive cant reach observed because glof peak flows are way too high

In [ ]:
with pm.Model() as model_glof:


    base_mu = pm.Normal("base_mu", mu=20, sigma=10)
    sigma = pm.HalfNormal("sigma", sigma=15)
    glof_effect = pm.Normal("glof_effect", mu=40, sigma=20)

    mu = pm.Deterministic("mu", base_mu + glof_indicator * glof_effect)


    y = pm.Normal("y", observed=actual_flow, mu=mu, sigma=sigma)


    prior_checks = pm.sample_prior_predictive(draws=1000, random_seed=42)

    # We must explicitly separate the creation of the trace and the addition of log-likelihood
    # when doing multiple extensions. The bug happens because sample_posterior_predictive tries to add
    # log_likelihood again if it thinks it's missing from the predictions group.


    trace_glof = pm.sample(draws=2000, tune=1000, chains=2, random_seed=42, progressbar=False)


    pm.compute_log_likelihood(trace_glof)


    pm.sample_posterior_predictive(trace_glof, extend_inferencedata=True, random_seed=42, progressbar=False)

In [ ]:
az.plot_ppc(prior_checks, group="prior", kind="kde", alpha=0.8)
plt.title("Prior Predictive")
plt.tight_layout()
plt.show()

# Tracing the atomic parameters, not the vectorized Deterministic

# if plot_trace explodes into overlapping lines/spaghetti

# It means you are asking PyMC to plot a vectorized tensor (like mu which contains 100 separate values for 100 separate days)
# rather than a unified atomic parameter (like base_mu or sigma which are single, global truths governing the system).

# but we wants overlapping (kde) plots in ppc plots

az.plot_trace(trace_glof, var_names=["base_mu", "glof_effect", "sigma"])
plt.tight_layout()
plt.show()

az.plot_ppc(trace_glof, group="posterior", kind="kde", alpha=0.8)
plt.title("Posterior Predictive")
plt.tight_layout()
plt.show()

print(az.summary(trace_glof, var_names=["base_mu", "glof_effect", "sigma"]))

# Trace proves the math; PPC proves the physics.

In [ ]:
print(az.compare(
    {"GLOF-Aware": trace_glof, "Naive": trace_naive},
    ic="loo"
))


# RULE: If d_loo > dse × 2, the higher-ranked model is MEANINGFULLY superior.

# from path we see that ignoring glof is statistically provable liability

# LOO-CV output

When you run az.compare(ic="loo"), PyMC evaluates how well each model predicts unseen data.

- **elpd_loo (Expected Log Predictive Density)**: The absolute score of the model's predictive power. Higher (less negative) is better. GLOF-Aware scores -298.

- **p_loo (Effective Parameters)**: The penalty for model complexity. GLOF-Aware uses ~3 parameters effectively; Naive model uses ~9 (it struggles to fit the extremes, artificially inflating its effective complexity).

- **elpd_diff (d_loo)**: The difference between the Top Ranked model and this model. GLOF-Aware is 0 (it is the top). Naive is 99.65. This is d_loo.

- **dse (Difference Standard Error)**: The uncertainty in that difference. For the Naive model, dse is 22.7.

- **weight**: The probability that the model is the "true" model in this set. GLOF-Aware gets 100% of the weight (1.0).